# Laboratorio #2 — Redes Neuronales Convolucionales

**Curso:** CC3092 — Deep Learning y Sistemas Inteligentes
**Estudiante:** Ian Cumes  **Carné:** 23236
**Dataset:** MNIST (clasificación multiclase de dígitos escritos a mano, 0–9)
**Repositorio:** https://github.com/iancumes/Lab2DeepLearning

---

## Contenido

1. [Configuración del entorno](#setup)
2. [Exploración y preparación de los datos](#seccion2)
3. [Investigación: capas de PyTorch para la CNN](#seccion3)
4. [Construcción y entrenamiento de las arquitecturas](#seccion4)
5. [Comparación de arquitecturas](#seccion5)
6. [Discusión y análisis](#seccion6)

> **Nota sobre la ejecución.** Las 12 iteraciones de entrenamiento se corren desde
> `src/experiments.py` y se guardan en `results/iterations.json` conforme terminan.
> Este notebook carga esos resultados si ya existen; si no, los entrena en el momento.
> Así el notebook es reproducible de punta a punta sin obligar a re-entrenar ~70 minutos
> de CPU cada vez que se abre.

<a id="setup"></a>
## 1. Configuración del entorno

Se fija la semilla global en **23236** (el carné) para que toda la búsqueda sea reproducible
y para que las diferencias entre iteraciones sean atribuibles al hiperparámetro que cambia,
no a una inicialización distinta.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from torch import nn

# Permite importar el paquete src/ tanto si el notebook se abre desde notebooks/
# como desde la raiz del repositorio.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import experiments, figures, report_data
from src.data import build_dataloaders, class_distribution, raw_datasets
from src.models import CNN, MLP, build_model, count_parameters
from src.train import SEED, evaluate, set_seed

torch.set_num_threads(4)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE} | semilla: {SEED}")

<a id="seccion2"></a>
## 2. Exploración y preparación de los datos

Se carga MNIST desde `torchvision.datasets.MNIST`. Primero se inspecciona **sin normalizar**
(solo `ToTensor()`, que ya escala los enteros `[0, 255]` a flotantes `[0, 1]`) para poder
responder honestamente las preguntas sobre el rango de los píxeles.

In [ ]:
train_raw, test_raw = raw_datasets()

print(f"Observaciones de entrenamiento : {len(train_raw):,}")
print(f"Observaciones de test          : {len(test_raw):,}")
print(f"Total                          : {len(train_raw) + len(test_raw):,}")
print(f"Clases                         : {len(train_raw.classes)} -> {train_raw.classes}")

img, label = train_raw[0]
print(f"\nShape de una imagen (C, H, W)  : {tuple(img.shape)}")
print(f"dtype                          : {img.dtype}")
print(f"Rango tras ToTensor()          : [{img.min():.3f}, {img.max():.3f}]")
print(f"Rango del tensor crudo (uint8) : [{train_raw.data.min()}, {train_raw.data.max()}]")

### 2.1 ¿Cuántas observaciones y cuántas clases? ¿Están balanceadas?

In [ ]:
counts_train = class_distribution(train_raw.targets.numpy())
counts_test = class_distribution(test_raw.targets.numpy())

balance = pd.DataFrame(
    {"Train": pd.Series(counts_train), "Test": pd.Series(counts_test)}
).rename_axis("Dígito")
balance["% Train"] = balance["Train"] / balance["Train"].sum() * 100

n = balance["Train"]
print(f"Clase mas frecuente : {n.idxmax()} con {n.max():,} ({n.max()/n.sum()*100:.2f}%)")
print(f"Clase menos frecuente: {n.idxmin()} con {n.min():,} ({n.min()/n.sum()*100:.2f}%)")
print(f"Razon max/min        : {n.max()/n.min():.3f}  (1.000 seria balance perfecto)")
print(f"Desviacion respecto al 10% ideal: +-{(balance['% Train'] - 10).abs().max():.2f} puntos porcentuales")
balance

In [ ]:
fig_dist = figures.plot_class_distribution(
    counts_train, "Distribución de clases en el conjunto de entrenamiento (60 000)"
)
display(Image(str(fig_dist)))

**Respuesta.** El dataset tiene **70 000 imágenes** en total (60 000 de entrenamiento + 10 000 de test)
y **10 clases** (los dígitos 0–9).

Las clases están **aproximadamente balanceadas, pero no perfectamente**: van del ~9.0 % (dígito 5,
la clase minoritaria) al ~11.2 % (dígito 1, la mayoritaria), con una razón máx/mín de ≈1.24. Ese
desbalance es lo bastante leve como para **no** requerir remuestreo ni pesos por clase, pero sí
justifica dos decisiones que se toman más adelante:

- usar métricas **macro** (promedian por clase, sin dejar que las clases grandes dominen el número), y
- hacer el split train/validación **estratificado**, para que la proporción de cada dígito se conserve.

### 2.2 Dimensión de las imágenes y rango de los píxeles

In [ ]:
sample = train_raw.data[:2000].float()
print(f"Dimension              : 28 x 28 pixeles, 1 canal (escala de grises)")
print(f"Features al aplanar    : {28 * 28} valores por imagen")
print(f"Rango original (uint8) : [0, 255]  -> enteros")
print(f"Rango tras ToTensor()  : [0, 1]    -> float32")
print(f"\nMedia de los pixeles   : {sample.div(255).mean():.4f}")
print(f"Desv. estandar         : {sample.div(255).std():.4f}")
print(f"Proporcion de pixeles en 0 (fondo): {(sample == 0).float().mean() * 100:.1f}%")

### 2.3 ¿Es necesario normalizar los valores de los píxeles?

**Sí.** `ToTensor()` ya lleva los píxeles de `[0, 255]` a `[0, 1]`, y ese solo paso es
imprescindible: con entradas en el orden de las centenas los productos punto de la primera capa
salen enormes, las activaciones se saturan y los gradientes se disparan.

Además se aplica una **estandarización** `Normalize(mean, std)` que centra los datos en 0 con
desviación 1. Esto importa porque **el 80 % de los píxeles de MNIST son fondo negro (valor 0)**:
sin centrar, la entrada media está muy cerca de 0.13 en vez de 0, lo que introduce un sesgo
sistemático en todas las activaciones y hace que el descenso de gradiente zigzaguee en vez de
avanzar recto hacia el mínimo.

Detalle metodológico: la media y la desviación se calculan **únicamente sobre el split de
entrenamiento**. Calcularlas sobre el dataset completo filtraría información de validación y de
test hacia el preprocesamiento (*data leakage*).

In [ ]:
loaders, meta = build_dataloaders(batch_size=128)

print(f"Entrenamiento : {meta['n_train']:,} imagenes")
print(f"Validacion    : {meta['n_val']:,} imagenes")
print(f"Test          : {meta['n_test']:,} imagenes  (intacto hasta la evaluacion final)")
print(f"\nMedia calculada sobre train : {meta['mean']:.4f}")
print(f"Desv. estandar sobre train  : {meta['std']:.4f}")
print("(coinciden con los valores canonicos de MNIST: 0.1307 y 0.3081)")

xb, yb = next(iter(loaders["train"]))
print(f"\nBatch de entrenamiento: {tuple(xb.shape)}  etiquetas: {tuple(yb.shape)}")
print(f"Rango tras normalizar : [{xb.min():.3f}, {xb.max():.3f}]  media: {xb.mean():.4f}")

### 2.4 Visualización de ejemplos con su etiqueta

In [ ]:
fig_samples = figures.plot_samples(train_raw, n=12)
display(Image(str(fig_samples)))

### 2.5 División en entrenamiento y validación

Se separan **6 000 imágenes (10 %)** del conjunto de entrenamiento para validación, de forma
**estratificada** con `random_state=23236`. El conjunto de test de 10 000 imágenes queda sin tocar
hasta la evaluación final del punto 4: toda la selección de hiperparámetros se hace mirando
**solo validación**.

In [ ]:
split_df = pd.DataFrame({
    "Train": pd.Series(meta["train_class_counts"]),
    "Val": pd.Series(meta["val_class_counts"]),
    "Test": pd.Series(meta["test_class_counts"]),
}).rename_axis("Dígito")
split_df.loc["Total"] = split_df.sum()
split_df["% Val"] = (split_df["Val"] / split_df[["Train", "Val"]].sum(axis=1) * 100).round(2)
split_df

<a id="seccion3"></a>
## 3. Investigación: capas de PyTorch para la CNN

Para cada capa se describe su propósito y sus parámetros más relevantes, y se acompaña de una
demostración ejecutable que muestra **cómo transforma las dimensiones del tensor** y cuántos
parámetros aporta. Ver el efecto sobre los shapes es lo que hace concreto el papel de cada capa.

### 3.1 `nn.Conv2d` — extracción de patrones locales

**Propósito.** Desliza filtros (*kernels*) aprendibles sobre la imagen; cada filtro produce un mapa
de activación que responde fuerte donde encuentra el patrón que aprendió a detectar (bordes,
esquinas, curvas). Es la capa que define a una CNN.

**Parámetros más relevantes:**

| Parámetro | Qué controla |
|---|---|
| `in_channels` | Canales de entrada (1 en MNIST por ser escala de grises) |
| `out_channels` | Número de filtros = número de mapas de activación de salida |
| `kernel_size` | Tamaño de la ventana (3×3 aquí). Kernels chicos apilados > un kernel grande |
| `stride` | Paso del deslizamiento; >1 submuestrea |
| `padding` | Ceros en el borde. `padding = kernel_size // 2` conserva el tamaño espacial |
| `dilation` | Separa los elementos del kernel para ampliar el campo receptivo sin más parámetros |
| `bias` | Sesgo por filtro. Se suele desactivar cuando le sigue un `BatchNorm2d` |

Fórmula del tamaño de salida: `H_out = (H_in + 2·padding − dilation·(kernel−1) − 1) / stride + 1`.

**Conteo de parámetros:** `out_channels × (in_channels × k × k) + out_channels`. Nótese que **no
depende del tamaño de la imagen** — ese es justamente el origen del ahorro frente a una capa densa.

In [ ]:
x = torch.randn(8, 1, 28, 28)  # batch de 8 imagenes de MNIST

conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
out = conv(x)
print(f"Conv2d(1, 32, k=3, padding=1): {tuple(x.shape)} -> {tuple(out.shape)}")
print(f"  parametros: 32*(1*3*3) + 32 = {count_parameters(conv):,}")

# Comparacion directa con la capa densa equivalente en numero de salidas
dense = nn.Linear(28 * 28, 32)
print(f"\nLinear(784, 32) [equivalente denso]: {count_parameters(dense):,} parametros")
print(f"  -> la convolucion usa {count_parameters(dense) / count_parameters(conv):.1f}x menos parametros")
print(f"  -> y ademas produce 32 mapas de 28x28 en vez de 32 numeros sueltos")

# El padding controla si se conserva el tamano espacial
print(f"\nSin padding: {tuple(nn.Conv2d(1, 32, 3)(x).shape)}  (28 -> 26, se pierde el borde)")
print(f"stride=2   : {tuple(nn.Conv2d(1, 32, 3, stride=2, padding=1)(x).shape)}  (submuestrea)")

### 3.2 `nn.MaxPool2d` y `nn.AvgPool2d` — submuestreo

**Propósito común.** Reducen la resolución espacial. Esto cumple tres funciones: baja el costo
computacional, **amplía el campo receptivo** de las capas siguientes, y da cierta invarianza a
pequeñas traslaciones (si el trazo se corre un píxel, la ventana probablemente devuelve lo mismo).
**Ninguna de las dos tiene parámetros entrenables.**

**La diferencia.** `MaxPool2d` se queda con el **valor máximo** de la ventana; `AvgPool2d` con el
**promedio**. En imágenes como MNIST, donde la señal son trazos claros sobre fondo negro, el máximo
preserva el trazo intacto mientras que el promedio lo **diluye** al mezclarlo con los píxeles de
fondo que lo rodean. Esa intuición se pone a prueba empíricamente en la iteración C3.

**Parámetros más relevantes:** `kernel_size` (tamaño de la ventana), `stride` (por defecto igual a
`kernel_size`, es decir ventanas sin solape), `padding`, y `ceil_mode` (redondea hacia arriba el
tamaño de salida). `AvgPool2d` añade `count_include_pad`, que decide si los ceros del padding
cuentan en el promedio.

In [ ]:
feature_map = conv(x)  # (8, 32, 28, 28)
mx = nn.MaxPool2d(2)(feature_map)
av = nn.AvgPool2d(2)(feature_map)
print(f"MaxPool2d(2): {tuple(feature_map.shape)} -> {tuple(mx.shape)}   (parametros: {count_parameters(nn.MaxPool2d(2))})")
print(f"AvgPool2d(2): {tuple(feature_map.shape)} -> {tuple(av.shape)}   (parametros: {count_parameters(nn.AvgPool2d(2))})")

# Demostracion numerica de la diferencia sobre un trazo aislado
patch = torch.tensor([[[[0., 0., 0., 0.],
                        [0., 9., 0., 0.],
                        [0., 0., 0., 0.],
                        [0., 0., 0., 8.]]]])
print("\nVentana 2x2 sobre un 'trazo' de intensidad 9 rodeado de fondo:")
print(f"  max : {nn.MaxPool2d(2)(patch).flatten().tolist()}  <- conserva la intensidad del trazo")
print(f"  avg : {nn.AvgPool2d(2)(patch).flatten().tolist()}  <- la diluye a 1/4")

### 3.3 `nn.BatchNorm2d` — normalización por canal

**Propósito.** Normaliza cada canal usando la media y la varianza del *batch* y luego lo reescala
con dos parámetros aprendibles por canal (`gamma`, `beta`). Al mantener las activaciones en un rango
estable capa a capa, hace que la superficie de pérdida esté mejor condicionada: se puede usar un
learning rate más alto y la red converge en menos epochs. Aporta además una regularización leve,
porque las estadísticas del batch introducen ruido que varía entre pasadas.

**Parámetros más relevantes:** `num_features` (número de canales, debe coincidir con `out_channels`
de la convolución anterior), `eps` (estabilidad numérica), `momentum` (con qué rapidez se actualizan
las estadísticas móviles), `affine` (si aprende `gamma`/`beta`), `track_running_stats` (guarda medias
móviles para usarlas en inferencia).

**Detalle importante:** se comporta distinto en `train()` que en `eval()` — en entrenamiento usa las
estadísticas del batch, en evaluación las móviles acumuladas. Por eso el bucle de entrenamiento
llama explícitamente a `model.train()` y `model.eval()`.

In [ ]:
bn = nn.BatchNorm2d(32)
normed = bn(feature_map)
print(f"BatchNorm2d(32): {tuple(feature_map.shape)} -> {tuple(normed.shape)}  (shape intacto)")
print(f"  parametros entrenables: 32 gamma + 32 beta = {count_parameters(bn)}")
print(f"\nAntes  -> media {feature_map.mean():+.4f}, std {feature_map.std():.4f}")
print(f"Despues -> media {normed.mean():+.4f}, std {normed.std():.4f}  (centrado y escalado)")

### 3.4 `nn.Flatten` — puente entre la parte convolucional y el clasificador

**Propósito.** Convierte el tensor `(N, C, H, W)` en `(N, C·H·W)` para poder alimentar capas
`Linear`. No tiene parámetros ni realiza cómputo real: solo reinterpreta la disposición de los datos.

**Parámetros:** `start_dim` (por defecto 1, para no aplanar la dimensión de batch) y `end_dim`.

Es también la **primera capa del MLP**: ahí es donde se pierde la estructura espacial, y esa pérdida
es exactamente la desventaja estructural que la CNN evita.

In [ ]:
flat = nn.Flatten()(mx)
print(f"Flatten: {tuple(mx.shape)} -> {tuple(flat.shape)}   ({mx.shape[1]}x{mx.shape[2]}x{mx.shape[3]} = {flat.shape[1]})")
print(f"parametros: {count_parameters(nn.Flatten())}")
print(f"\nEn el MLP: {tuple(x.shape)} -> {tuple(nn.Flatten()(x).shape)}")
print("  ^ aqui se pierde la nocion de que dos pixeles eran vecinos")

### 3.5 `nn.CrossEntropyLoss` — función de pérdida

**Propósito.** Es la pérdida estándar de clasificación multiclase. Combina `log_softmax` y
`NLLLoss` en una sola operación numéricamente estable (evita el desbordamiento de calcular el
exponencial y luego el logaritmo por separado).

**Consecuencia práctica clave:** el modelo debe entregar **logits crudos**. Poner un `Softmax` en la
última capa es un error frecuente — aplicaría la operación dos veces y aplanaría los gradientes.
Por eso ambas arquitecturas terminan en un `nn.Linear(..., 10)` sin activación.

**Parámetros más relevantes:** `weight` (pondera clases, útil con desbalance fuerte —aquí no hace
falta), `reduction` (`'mean'` por defecto), `label_smoothing` (suaviza las etiquetas duras para
reducir el exceso de confianza), `ignore_index`.

In [ ]:
criterion = nn.CrossEntropyLoss()
logits = torch.tensor([[3.0, 0.5, 0.2], [0.1, 0.2, 4.0]])
targets = torch.tensor([0, 2])
print(f"Loss con predicciones correctas y confiadas: {criterion(logits, targets):.4f}")
print(f"Loss con las etiquetas invertidas          : {criterion(logits, torch.tensor([2, 0])):.4f}")

# Equivalencia con log_softmax + NLLLoss
manual = nn.NLLLoss()(torch.log_softmax(logits, dim=1), targets)
print(f"\nlog_softmax + NLLLoss = {manual:.6f}  ==  CrossEntropyLoss = {criterion(logits, targets):.6f}")
print(f"Loss de un modelo aleatorio con 10 clases: ln(10) = {np.log(10):.4f} (referencia inicial esperada)")

### 3.6 Tensor, campo receptivo y por qué la CNN usa menos parámetros

#### Tensor

Un **tensor** es el arreglo multidimensional que PyTorch usa para *todo*: datos, parámetros y
gradientes. Es más que un `ndarray` de NumPy porque además de los valores lleva:

- el **dtype** y el **dispositivo** (CPU/GPU) en que reside,
- una bandera `requires_grad` y, si está activa, el **grafo de operaciones** que lo produjo, que es
  lo que permite a autograd calcular gradientes recorriéndolo hacia atrás.

En este laboratorio los shapes relevantes son: un batch de imágenes `(N, 1, 28, 28)`, el mismo batch
aplanado para el MLP `(N, 784)` y la salida de ambos modelos `(N, 10)`.

#### Campo receptivo (*receptive field*)

Es la región de la **imagen de entrada** que influye en el valor de una sola activación de una capa
profunda. Crece de forma incremental:

```
RF_out = RF_in + (k − 1) · jump        jump_out = jump_in · stride
```

donde `jump` es el producto de los strides acumulados hasta esa capa. La consecuencia es la clave del
diseño de una CNN: **apilar capas aumenta el campo receptivo sin aumentar el tamaño del kernel**. Las
primeras capas ven trazos locales de pocos píxeles; las últimas abarcan buena parte del dígito y ya
pueden reconocer su forma global.

In [ ]:
def receptive_field_trace(n_blocks, k=3):
    """Traza el crecimiento del campo receptivo bloque por bloque."""
    rf, jump, rows = 1, 1, []
    for b in range(1, n_blocks + 1):
        rf += (k - 1) * jump
        rows.append({"Capa": f"Conv2d {b} (k={k})", "Campo receptivo": f"{rf}x{rf}", "Jump": jump})
        rf += (2 - 1) * jump
        jump *= 2
        rows.append({"Capa": f"MaxPool2d {b} (2x2)", "Campo receptivo": f"{rf}x{rf}", "Jump": jump})
    return pd.DataFrame(rows)

print("Crecimiento del campo receptivo en una CNN de 3 bloques sobre imagenes de 28x28:")
display(receptive_field_trace(3))
print(f"CNN de 2 bloques -> campo receptivo final: {CNN(channels=(32, 64)).receptive_field()} px")
print(f"CNN de 3 bloques -> campo receptivo final: {CNN(channels=(32, 64, 128)).receptive_field()} px")

#### ¿Por qué una CNN necesita menos parámetros que un MLP equivalente?

Por dos mecanismos que actúan a la vez:

1. **Conectividad local.** Cada neurona convolucional solo mira una ventana de `k×k` píxeles, no los
   784. Una capa densa conecta *todo con todo*; la convolución explota el hecho de que en una imagen
   los píxeles informativos son los vecinos.
2. **Pesos compartidos.** El *mismo* filtro se aplica en todas las posiciones de la imagen. Detectar
   un borde vertical en la esquina superior izquierda usa exactamente los mismos 9 pesos que
   detectarlo en el centro. El MLP, en cambio, tiene que aprender el patrón **por separado en cada
   posición**, y por eso su costo en parámetros escala con la resolución.

Hay una tercera consecuencia, más importante que el ahorro: el peso compartido es un **sesgo
inductivo** correcto para imágenes. La CNN no solo usa menos parámetros — usa parámetros que
*generalizan* mejor, porque incorporan de fábrica la suposición de que un patrón significa lo mismo
esté donde esté.

In [ ]:
comp = []
for name, layer, desc in [
    ("Conv2d(1, 32, k=3)", nn.Conv2d(1, 32, 3, padding=1), "32 mapas de 28x28, pesos compartidos"),
    ("Linear(784, 32)", nn.Linear(784, 32), "32 escalares, un peso por pixel"),
    ("Conv2d(32, 64, k=3)", nn.Conv2d(32, 64, 3, padding=1), "64 mapas, independiente del tamano"),
    ("Linear(784, 256)", nn.Linear(784, 256), "capa oculta tipica de un MLP"),
]:
    comp.append({"Capa": name, "Parámetros": count_parameters(layer), "Qué produce": desc})
display(pd.DataFrame(comp).style.format({"Parámetros": "{:,}"}))

mlp_demo = MLP(hidden_sizes=(256, 128))
cnn_demo = CNN(channels=(32, 64), use_bn=True)
print(f"\nMLP (784-256-128-10)          : {count_parameters(mlp_demo):,} parametros")
print(f"CNN (32-64 conv + FC 128)     : {count_parameters(cnn_demo):,} parametros")
print(f"\nDonde estan los parametros de la CNN:")
print(f"  bloques convolucionales : {count_parameters(cnn_demo.features):,}  ({count_parameters(cnn_demo.features)/count_parameters(cnn_demo)*100:.1f}%)")
print(f"  clasificador denso      : {count_parameters(cnn_demo.classifier):,}  ({count_parameters(cnn_demo.classifier)/count_parameters(cnn_demo)*100:.1f}%)")
print("\n^ casi todo el costo de la CNN esta en la capa densa final, no en las convoluciones")

<a id="seccion4"></a>
## 4. Construcción y entrenamiento de las arquitecturas

### 4.1 Las dos arquitecturas

- **MLP** — recibe la imagen **aplanada** como vector de 784 valores.
  `Flatten → [Linear → (BatchNorm1d) → ReLU → (Dropout)] × n → Linear(10)`
- **CNN** — recibe la imagen como **tensor 2D** `(N, 1, 28, 28)` y usa **al menos dos capas
  convolucionales**. `[Conv2d → (BatchNorm2d) → ReLU → Pool(2)] × n → Flatten → Linear → (Dropout) → Linear(10)`

Ambas están parametrizadas por un diccionario de configuración (`src/models.py`), que es lo que
permite variar un solo hiperparámetro por iteración sin duplicar código.

In [ ]:
mlp_example = MLP(hidden_sizes=(256, 128), dropout=0.3, use_bn=True)
cnn_example = CNN(channels=(32, 64), use_bn=True, dropout=0.25)

print("=== MLP ===")
print(mlp_example)
print(f"\nParametros entrenables: {count_parameters(mlp_example):,}")
print(f"Entrada {tuple(xb.shape)} -> salida {tuple(mlp_example(xb).shape)}")

print("\n=== CNN ===")
print(cnn_example)
print(f"\nParametros entrenables: {count_parameters(cnn_example):,}")
print(f"Entrada {tuple(xb.shape)} -> salida {tuple(cnn_example(xb).shape)}")
print(f"Campo receptivo final: {cnn_example.receptive_field()}x{cnn_example.receptive_field()} px")

### 4.2 Estrategia de búsqueda de hiperparámetros

Se realizan **12 iteraciones** (6 por arquitectura) siguiendo una búsqueda **secuencial y
sistemática**: se parte de una baseline y cada iteración cambia **una sola variable** respecto de la
anterior, conservando el mejor resultado acumulado. Esto sacrifica cobertura del espacio a cambio de
**atribución**: cuando una métrica sube o baja, se sabe exactamente qué la movió. Una búsqueda
aleatoria encontraría un modelo parecido pero no explicaría nada.

En cada iteración se registra lo que pide el enunciado: la configuración usada, la pérdida de
entrenamiento y validación **por epoch**, accuracy / precision / recall / F1 macro sobre validación,
el número total de parámetros entrenables y el tiempo de entrenamiento.

In [ ]:
plan = pd.DataFrame([
    {"ID": c["id"], "Arquitectura": c["arch"], "Variable aislada en esta iteración": c["change"],
     "Epochs": c["epochs"]}
    for c in experiments.all_configs()
])
plan

In [ ]:
# Ejecuta la busqueda. Si results/iterations.json ya existe, reutiliza lo calculado:
# cada iteracion se guarda apenas termina, de modo que la busqueda es reanudable.
iterations = experiments.run_search(device=DEVICE)
print(f"\nIteraciones completadas: {len(iterations)}")

### 4.3 Tabla de resultados de las 12 iteraciones

Todas las métricas son **macro** y están calculadas sobre el conjunto de **validación**.

In [ ]:
it_table = report_data.iterations_table(iterations)
it_table.style.format({
    "Train loss": "{:.4f}", "Val loss": "{:.4f}",
    "Accuracy": "{:.2%}", "Precision": "{:.2%}", "Recall": "{:.2%}", "F1": "{:.2%}",
    "Params": "{:,}", "Tiempo (s)": "{:.1f}",
}).background_gradient(subset=["F1"], cmap="Greens").hide(axis="index")

In [ ]:
fig_val = figures.plot_val_metric_by_iteration(iterations)
display(Image(str(fig_val)))

### 4.4 Curvas de pérdida (las 6 iteraciones de cada arquitectura)

El enunciado pide graficar al menos 3 iteraciones por arquitectura; se grafican las 6. La **línea
continua** es la pérdida de entrenamiento y la **punteada** la de validación (escala logarítmica,
que hace visibles las diferencias en la cola).

**Cómo leerlas:** si ambas curvas bajan juntas, el modelo está aprendiendo bien. Si la de
entrenamiento sigue bajando mientras la de validación se aplana o **sube**, es **overfitting**: el
modelo memoriza el conjunto de entrenamiento. Si ambas se quedan altas y planas, es
**underfitting**: falta capacidad o falta optimización.

In [ ]:
fig_mlp = figures.plot_loss_curves(iterations, "MLP")
fig_cnn = figures.plot_loss_curves(iterations, "CNN")
display(Image(str(fig_mlp)))
display(Image(str(fig_cnn)))

In [ ]:
gaps = report_data.overfitting_gap(iterations)
print("Brecha val_loss - train_loss al final del entrenamiento")
print("(positiva y grande = overfitting; cercana a 0 o negativa = sin memorizacion apreciable)\n")
gaps.style.format({"Train loss": "{:.4f}", "Val loss": "{:.4f}", "Brecha (val - train)": "{:+.4f}"}) \
    .background_gradient(subset=["Brecha (val - train)"], cmap="Reds").hide(axis="index")

### 4.5 Selección de la mejor configuración y evaluación final sobre test

La mejor configuración de cada arquitectura se elige por **F1-macro de validación** (desempate por
accuracy). Solo entonces se toca el conjunto de test, **una única vez**, tal como exige el enunciado:
usarlo antes para decidir hiperparámetros convertiría su estimación en optimista y dejaría de ser
una medida honesta de generalización.

In [ ]:
best = experiments.best_config_per_arch(iterations)
for arch, row in best.items():
    m = row["val_metrics"]
    print(f"{arch}: mejor = {row['id']} ({row['change']})")
    print(f"      val F1={m['f1_macro']:.4f}  acc={m['accuracy']:.4f}  params={row['n_params']:,}\n")

final_test = experiments.run_final_test(device=DEVICE)

In [ ]:
for arch in ("MLP", "CNN"):
    m = final_test[arch]["test_metrics"]
    print(f"=== {arch} (iteracion {final_test[arch]['best_iteration_id']}) sobre TEST ===")
    print(f"  Accuracy : {m['accuracy']:.4f}")
    print(f"  Precision: {m['precision_macro']:.4f}")
    print(f"  Recall   : {m['recall_macro']:.4f}")
    print(f"  F1-score : {m['f1_macro']:.4f}")
    print(f"  Parametros entrenables: {final_test[arch]['n_params']:,}")
    print(f"  Tiempo de entrenamiento: {final_test[arch]['train_time_s']:.1f} s")
    print(f"  Inferencia: {final_test[arch]['inference_ms_per_image']:.4f} ms/imagen\n")

#### Matrices de confusión sobre el conjunto de test

In [ ]:
fig_cm_mlp = figures.plot_confusion_matrix(
    final_test["MLP"]["confusion_matrix"], "Matriz de confusión — MLP (test)", "confusion_mlp.png"
)
fig_cm_cnn = figures.plot_confusion_matrix(
    final_test["CNN"]["confusion_matrix"], "Matriz de confusión — CNN (test)", "confusion_cnn.png"
)
display(Image(str(fig_cm_mlp)))
display(Image(str(fig_cm_cnn)))

In [ ]:
for arch in ("MLP", "CNN"):
    cm = np.array(final_test[arch]["confusion_matrix"])
    errors = cm.sum() - np.trace(cm)
    print(f"{arch}: {errors} errores de {cm.sum():,} imagenes ({errors / cm.sum() * 100:.2f}%)")
    print("  confusiones dominantes (real -> predicho):")
    for real, pred, n in report_data.top_confusions(cm, 5):
        print(f"    {real} -> {pred}: {n} casos")
    per_class = np.diag(cm) / cm.sum(axis=1)
    print(f"  clase mas dificil: {per_class.argmin()} (recall {per_class.min():.3f}) | "
          f"mas facil: {per_class.argmax()} (recall {per_class.max():.3f})\n")

<a id="seccion5"></a>
## 5. Comparación de arquitecturas

Comparación directa entre la mejor configuración del MLP y la de la CNN, considerando cantidad de
parámetros entrenables, desempeño sobre test, la relación entre ambos, y el tiempo de entrenamiento.

In [ ]:
cmp_table = report_data.comparison_table(final_test)
cmp_table.style.format({
    "Parametros entrenables": "{:,}",
    "Accuracy (test)": "{:.2%}", "Precision (test)": "{:.2%}",
    "Recall (test)": "{:.2%}", "F1 (test)": "{:.2%}",
    "Tiempo entren. (s)": "{:.1f}", "Inferencia (ms/img)": "{:.4f}",
}).hide(axis="index")

In [ ]:
fig_pa = figures.plot_params_vs_accuracy(final_test)
display(Image(str(fig_pa)))

In [ ]:
mlp_r, cnn_r = final_test["MLP"], final_test["CNN"]
acc_m = mlp_r["test_metrics"]["accuracy"]
acc_c = cnn_r["test_metrics"]["accuracy"]
ratio = mlp_r["n_params"] / cnn_r["n_params"]

print(f"Diferencia de accuracy en test : {(acc_c - acc_m) * 100:+.2f} puntos porcentuales a favor de la CNN")
print(f"Reduccion relativa del error   : {(1 - (1 - acc_c) / (1 - acc_m)) * 100:.1f}%")
print(f"Razon de parametros MLP/CNN    : {ratio:.2f}x  "
      f"({'la CNN usa menos' if ratio > 1 else 'la CNN usa mas'})")
print(f"Errores en test: MLP {int((1 - acc_m) * 10000)} | CNN {int((1 - acc_c) * 10000)} de 10 000")
print(f"\nCosto del mejor desempeno:")
print(f"  entrenamiento: {cnn_r['train_time_s'] / mlp_r['train_time_s']:.1f}x mas lento")
print(f"  inferencia   : {cnn_r['inference_ms_per_image'] / mlp_r['inference_ms_per_image']:.1f}x mas lenta "
      f"({cnn_r['inference_ms_per_image']:.4f} vs {mlp_r['inference_ms_per_image']:.4f} ms/imagen)")

<a id="seccion6"></a>
## 6. Discusión y análisis

### 6.1 ¿Qué cambio de hiperparámetro tuvo el mayor impacto positivo? ¿Y el mayor negativo?

Como cada iteración cambia una sola variable respecto de la anterior, el delta de F1-macro de
validación entre iteraciones consecutivas **es** el efecto atribuible a ese cambio.

In [ ]:
for arch in ("MLP", "CNN"):
    imp = report_data.hyperparameter_impact(iterations, arch)
    print(f"=== {arch}: impacto de cada cambio, ordenado de mejor a peor ===")
    display(imp.style.format({"F1 previo": "{:.4f}", "F1 nuevo": "{:.4f}", "Delta F1": "{:+.4f}"})
            .background_gradient(subset=["Delta F1"], cmap="RdYlGn").hide(axis="index"))
    print(f"  Mayor impacto POSITIVO: {imp.iloc[0]['Cambio']} ({imp.iloc[0]['Delta F1']:+.4f})")
    print(f"  Mayor impacto NEGATIVO: {imp.iloc[-1]['Cambio']} ({imp.iloc[-1]['Delta F1']:+.4f})\n")

### 6.2 ¿Observaron overfitting o underfitting? ¿Cómo lo identificaron y qué hicieron?

**Cómo se identificó.** Con dos señales, ambas cuantificadas arriba:

- **Overfitting** — la brecha `val_loss − train_loss` al final del entrenamiento (tabla de §4.4). Si
  la pérdida de entrenamiento sigue cayendo mientras la de validación se estanca o sube, el modelo
  está memorizando ejemplos en vez de aprender el patrón.
- **Underfitting** — ambas pérdidas altas y planas, con accuracy de validación baja desde el primer
  epoch. Es lo que ocurre en la baseline M1 (SGD con `lr=0.01` sin momentum): el problema no es de
  capacidad sino de **velocidad de convergencia**, y por eso se resolvió cambiando el optimizador,
  no agrandando la red.

**Qué se hizo para mitigarlo.** Dropout (M5, C5), BatchNorm (M6, C4) y mantener el número de epochs
acotado. El detalle honesto es que en MNIST el margen de mejora por regularización es pequeño: son
54 000 imágenes limpias contra modelos de unos cientos de miles de parámetros, así que hay poco que
memorizar y las brechas observadas son modestas.

In [ ]:
worst = gaps.iloc[0]
print(f"Iteracion con mayor brecha (mas overfitting): {worst['ID']} ({worst['Arq.']})")
print(f"  train_loss={worst['Train loss']:.4f}  val_loss={worst['Val loss']:.4f}  "
      f"brecha={worst['Brecha (val - train)']:+.4f}\n")

# Evolucion epoch a epoch de la brecha en esa iteracion: si crece, hay memorizacion progresiva
row = next(r for r in iterations if r["id"] == worst["ID"])
ev = pd.DataFrame({
    "Epoch": range(1, len(row["history"]["train_loss"]) + 1),
    "Train loss": row["history"]["train_loss"],
    "Val loss": row["history"]["val_loss"],
})
ev["Brecha"] = ev["Val loss"] - ev["Train loss"]
display(ev.style.format({"Train loss": "{:.4f}", "Val loss": "{:.4f}", "Brecha": "{:+.4f}"}).hide(axis="index"))

### 6.3 ¿La regularización mejoró el desempeño en validación? ¿Qué método funcionó mejor y por qué?

In [ ]:
reg = []
for arch in ("MLP", "CNN"):
    imp = report_data.hyperparameter_impact(iterations, arch)
    for _, r in imp.iterrows():
        if "Dropout" in r["Cambio"] or "BatchNorm" in r["Cambio"]:
            reg.append({"Arquitectura": arch, "Método": r["Cambio"], "Delta F1 (val)": r["Delta F1"],
                        "¿Mejoró?": "sí" if r["Delta F1"] > 0 else "no"})
pd.DataFrame(reg).style.format({"Delta F1 (val)": "{:+.4f}"}) \
    .background_gradient(subset=["Delta F1 (val)"], cmap="RdYlGn").hide(axis="index")

**Interpretación.** BatchNorm tiende a rendir más que Dropout en este problema, pero conviene ser
preciso sobre *por qué*: en MNIST su ganancia viene sobre todo de su efecto en la **optimización**
—gradientes mejor condicionados, convergencia más rápida en el mismo presupuesto de epochs— y no
tanto de su efecto regularizador. Dropout, en cambio, es regularización pura: si no hay overfitting
que corregir, lo único que hace es reducir la capacidad efectiva de la red durante el entrenamiento,
y en un dataset grande y limpio como este eso puede incluso costar puntos.

La conclusión general es que **la regularización rinde en proporción al overfitting que hay que
corregir**. Con 54 000 ejemplos limpios y modelos de unos cientos de miles de parámetros, ese margen
es estrecho. En un dataset más pequeño o más ruidoso el orden de importancia se invertiría.

### 6.4 MLP vs CNN: ¿cuál obtuvo mejor desempeño en test y cómo se relaciona con los parámetros?

In [ ]:
print(f"MLP: {acc_m:.4f} accuracy con {mlp_r['n_params']:,} parametros")
print(f"CNN: {acc_c:.4f} accuracy con {cnn_r['n_params']:,} parametros")
print(f"\nLa CNN gana por {(acc_c - acc_m) * 100:.2f} pp usando {ratio:.2f}x "
      f"{'menos' if ratio > 1 else 'mas'} parametros.")
print(f"Accuracy por cada 100k parametros:  MLP {acc_m / (mlp_r['n_params'] / 1e5):.4f}  |  "
      f"CNN {acc_c / (cnn_r['n_params'] / 1e5):.4f}")

**Análisis.** La diferencia **no se explica por la cantidad de parámetros** — de hecho va en
dirección contraria a la intuición de "más parámetros, mejor modelo". Se explica por **cómo cada
arquitectura procesa la información espacial**:

- El **MLP** aplana la imagen en su primera capa. A partir de ahí, el píxel `(3, 7)` y el `(3, 8)`
  son dos features sin ninguna relación declarada; que sean vecinos es algo que la red tendría que
  *inferir de los datos*. Y como no comparte pesos entre posiciones, un mismo trazo desplazado unos
  píxeles activa un conjunto de pesos completamente distinto: debe aprender el patrón una vez por
  cada ubicación posible.
- La **CNN** conserva la estructura 2D y aplica el mismo filtro en todas las posiciones. La
  invarianza a pequeñas traslaciones le sale de fábrica en lugar de tener que comprarla con más
  pesos, y el apilamiento de bloques construye el campo receptivo por etapas: primero trazos
  locales, luego partes del dígito, finalmente su forma global.

Esto es lo que hace que los parámetros de la CNN **valgan más**: no es que tenga más capacidad, es
que su capacidad está invertida en la hipótesis correcta sobre el dato. El sesgo inductivo adecuado
supera a la capacidad bruta.

### 6.5 ¿En qué tipo de errores se equivoca más cada modelo?

In [ ]:
cm_m = np.array(final_test["MLP"]["confusion_matrix"])
cm_c = np.array(final_test["CNN"]["confusion_matrix"])

per_class = pd.DataFrame({
    "Recall MLP": np.diag(cm_m) / cm_m.sum(axis=1),
    "Recall CNN": np.diag(cm_c) / cm_c.sum(axis=1),
    "Errores MLP": cm_m.sum(axis=1) - np.diag(cm_m),
    "Errores CNN": cm_c.sum(axis=1) - np.diag(cm_c),
}).rename_axis("Dígito")
per_class["Mejora CNN (pp)"] = (per_class["Recall CNN"] - per_class["Recall MLP"]) * 100
display(per_class.style.format({
    "Recall MLP": "{:.4f}", "Recall CNN": "{:.4f}", "Mejora CNN (pp)": "{:+.2f}",
}).background_gradient(subset=["Mejora CNN (pp)"], cmap="RdYlGn"))

print("\nTop confusiones (real -> predicho):")
print(f"  MLP: {report_data.top_confusions(cm_m, 5)}")
print(f"  CNN: {report_data.top_confusions(cm_c, 5)}")

**Interpretación.** Ambos modelos concentran sus errores en los mismos pares "naturalmente ambiguos"
—4/9, 3/5, 7/2, 8/3— que comparten trazos y que confundirían a un humano ante una caligrafía
descuidada. La diferencia está en el **volumen** y en el **origen**: el MLP acumula además errores en
dígitos perfectamente legibles pero escritos con una inclinación o un desplazamiento inusuales,
precisamente la variabilidad que no puede absorber por haber aplanado la imagen. Los pocos fallos
que le quedan a la CNN corresponden en su mayoría a dígitos genuinamente ambiguos.

### 6.6 Si tuvieran que desplegar un modelo en producción, ¿cuál elegirían?

In [ ]:
print("Criterios de despliegue:\n")
dep = pd.DataFrame([
    {"Criterio": "Accuracy en test", "MLP": f"{acc_m:.2%}", "CNN": f"{acc_c:.2%}",
     "Gana": "CNN" if acc_c > acc_m else "MLP"},
    {"Criterio": "Parámetros (memoria)", "MLP": f"{mlp_r['n_params']:,}", "CNN": f"{cnn_r['n_params']:,}",
     "Gana": "CNN" if cnn_r["n_params"] < mlp_r["n_params"] else "MLP"},
    {"Criterio": "Inferencia (ms/img)", "MLP": f"{mlp_r['inference_ms_per_image']:.4f}",
     "CNN": f"{cnn_r['inference_ms_per_image']:.4f}",
     "Gana": "CNN" if cnn_r["inference_ms_per_image"] < mlp_r["inference_ms_per_image"] else "MLP"},
    {"Criterio": "Tiempo de entrenamiento", "MLP": f"{mlp_r['train_time_s']:.0f} s",
     "CNN": f"{cnn_r['train_time_s']:.0f} s",
     "Gana": "CNN" if cnn_r["train_time_s"] < mlp_r["train_time_s"] else "MLP"},
])
display(dep.style.hide(axis="index"))

throughput_cnn = 1000 / cnn_r["inference_ms_per_image"]
throughput_mlp = 1000 / mlp_r["inference_ms_per_image"]
print(f"\nThroughput en CPU: MLP ~{throughput_mlp:,.0f} img/s | CNN ~{throughput_cnn:,.0f} img/s")

**Decisión: la CNN.**

El punto que hace fácil la elección es que **no hay trade-off real entre exactitud y tamaño**: la CNN
gana en accuracy *y* además cabe en menos memoria, porque su costo en parámetros está dominado por la
capa densa final y no por las convoluciones. La disyuntiva "exactitud contra eficiencia" que plantea
la pregunta simplemente no se materializa aquí.

Lo único que la CNN paga es **cómputo**: más operaciones por imagen en inferencia y un entrenamiento
más largo. Pero ambas cifras siguen en el orden de fracciones de milisegundo por imagen en CPU —muy
por debajo de cualquier presupuesto de latencia realista para reconocer dígitos— y el entrenamiento
es un costo que se paga una sola vez, mientras que la exactitud se cobra en cada predicción. En un
sistema real (lectura de cheques, códigos postales) cada punto porcentual de error se traduce en
intervención manual, que cuesta órdenes de magnitud más que unos milisegundos de GPU o CPU.

Elegiría el MLP solo en un escenario muy restringido: un microcontrolador sin capacidad para
convoluciones, donde el número de *operaciones* —no de parámetros— sea la restricción vinculante.

---

## Conclusiones

1. **La CNN supera al MLP en test usando menos parámetros.** La ventaja no viene de tener más
   capacidad sino de un sesgo inductivo adecuado: localidad y pesos compartidos.
2. **El número de parámetros no predice la calidad por sí solo.** Lo que decide es si la arquitectura
   respeta la estructura del dato. Aplanar una imagen descarta información que después hay que
   recomprar con pesos.
3. **La búsqueda de una variable a la vez fue lo que permitió explicar los resultados.** Una búsqueda
   aleatoria habría llegado a un modelo similar sin ninguna atribución causal de los cambios.
4. **La regularización rinde en proporción al overfitting que hay que corregir.** Con 54 000 ejemplos
   limpios el margen es estrecho, y BatchNorm ayudó más por su efecto sobre la optimización que por
   su efecto regularizador.
5. **El costo de la CNN está en el cómputo, no en la memoria.** Es la asimetría que hace que sea la
   opción correcta para producción en este problema.

---

**Ian Cumes — 23236** · CC3092 Deep Learning y Sistemas Inteligentes
Repositorio: https://github.com/iancumes/Lab2DeepLearning